# OTIUM — EN narrative fill (Colab)

Fill **English** place text for city guides.

**Input:** upload `en_narrative_fill.json` (or `en_narrative_fill_moscow.json`).

**Output:** download `en_narrative_fill_results.jsonl` →
`translations/results/en_narrative_fill_results.jsonl` on your PC.

**Apply on PC:** `python scripts/apply_en_narrative_fill_results.py`

In [ ]:
# --- Configuration ---
PROVIDER = "gemini"  # "gemini" | "openai"
# Use a current model id from ai.google.dev (avoid gemini-2.0-flash if 400)
GEMINI_MODEL = "gemini-2.5-flash"
OPENAI_MODEL = "gpt-4o-mini"

INPUT_JSON = "en_narrative_fill.json"
OUTPUT_JSONL = "en_narrative_fill_results.jsonl"
CHECKPOINT_EVERY = 10
DELAY_SEC = 1.0
LIMIT = None  # e.g. 3 for smoke test
START_AT = 0
RESUME = True

In [ ]:
!pip -q -U google-genai openai

In [ ]:
import json
import re
import time
from getpass import getpass
from pathlib import Path

from google import genai
from google.colab import files
from openai import OpenAI

if not Path(INPUT_JSON).is_file():
    print("Upload", INPUT_JSON)
    uploaded = files.upload()
    if INPUT_JSON not in uploaded:
        raise FileNotFoundError("Expected {} after upload".format(INPUT_JSON))

jobs = json.loads(Path(INPUT_JSON).read_text(encoding="utf-8"))
print("Loaded {} job(s)".format(len(jobs)))

In [ ]:
if PROVIDER == "gemini":
    api_key = getpass("Gemini API key (aistudio.google.com): ")
    _gemini_client = genai.Client(api_key=api_key)
elif PROVIDER == "openai":
    api_key = getpass("OpenAI API key: ")
    _openai = OpenAI(api_key=api_key)
else:
    raise ValueError("PROVIDER must be gemini or openai")

In [ ]:
# Smoke test — run this before the 431-job loop
if PROVIDER == "gemini":
    test = _gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Return JSON only: {"ok": true}',
    )
    print("Model:", GEMINI_MODEL)
    print("Test response:", (test.text or "")[:120])
else:
    test = _openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": 'JSON: {"ok": true}'}],
        max_tokens=20,
    )
    print("Model:", OPENAI_MODEL)
    print("Test response:", test.choices[0].message.content)

In [ ]:
EXPECTED_KEYS = (
    "description_en",
    "history_en",
    "significance_en",
    "facts_en",
    "stories_en",
)


def extract_json(text: str) -> dict | None:
    text = (text or "").strip()
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1)
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        obj = json.loads(text[start : end + 1])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None


def call_llm(prompt: str) -> str:
    """Plain text call; prompt already asks for JSON (no response_mime_type)."""
    if PROVIDER == "gemini":
        resp = _gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
        )
        return (resp.text or "").strip()
    resp = _openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.35,
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()


def normalize_result(job_id: str, parsed: dict) -> dict:
    row = {"id": job_id, "ok": True}
    for key in EXPECTED_KEYS:
        if key not in parsed:
            continue
        val = parsed[key]
        if key in ("facts_en", "stories_en") and isinstance(val, list):
            row[key] = [str(x).strip() for x in val if str(x).strip()]
        else:
            row[key] = str(val).strip() if val is not None else ""
    if not any(row.get(k) for k in EXPECTED_KEYS[:3]) and not row.get("facts_en"):
        row["ok"] = False
        row["error"] = "empty payload"
    return row

In [ ]:
def load_done(path: str) -> set[str]:
    p = Path(path)
    if not p.is_file():
        return set()
    done = set()
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get("ok"):
            done.add(str(row.get("id") or ""))
    return done


def append_jsonl(path: str, rows: list[dict]) -> None:
    with Path(path).open("a", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


def format_error(exc: BaseException) -> str:
    msg = str(exc)
    if len(msg) > 400:
        return msg[:400]
    return msg


done_ids = load_done(OUTPUT_JSONL) if RESUME else set()
subset = jobs[START_AT:]
if LIMIT is not None:
    subset = subset[: int(LIMIT)]

pending = [j for j in subset if j.get("id") not in done_ids]
print("Pending:", len(pending), "(skipped done:", len(done_ids), ")")

buffer: list[dict] = []
ok_count = err_count = 0
first_error: str | None = None

for i, job in enumerate(pending, start=1):
    jid = str(job.get("id") or "")
    prompt = str(job.get("prompt") or "")
    try:
        raw = call_llm(prompt)
        parsed = extract_json(raw)
        if not parsed:
            row = {
                "id": jid,
                "ok": False,
                "error": "json_parse_failed",
                "raw": raw[:500],
            }
            err_count += 1
        else:
            row = normalize_result(jid, parsed)
            if row.get("ok"):
                ok_count += 1
            else:
                err_count += 1
    except Exception as exc:
        err_text = format_error(exc)
        if first_error is None:
            first_error = err_text
            print("First API error:", first_error)
        row = {"id": jid, "ok": False, "error": err_text}
        err_count += 1

    buffer.append(row)
    if i % CHECKPOINT_EVERY == 0:
        append_jsonl(OUTPUT_JSONL, buffer)
        buffer.clear()
        print("[{}/{}] ok={} err={}".format(
            i, len(pending), ok_count, err_count,
        ))
        if err_count == i and first_error:
            print("All failed — fix model/key before continuing.")
            break
    if DELAY_SEC:
        time.sleep(float(DELAY_SEC))

if buffer:
    append_jsonl(OUTPUT_JSONL, buffer)

print("Done. ok={} err={} -> {}".format(
    ok_count, err_count, OUTPUT_JSONL,
))

In [ ]:
from google.colab import files

if Path(OUTPUT_JSONL).is_file():
    print("Download:", OUTPUT_JSONL)
    files.download(OUTPUT_JSONL)
else:
    print("No output file yet.")

## If you get HTTP 400

1. **Re-upload** the updated notebook from the repo.
2. Set `GEMINI_MODEL = "gemini-2.5-flash"` (or `gemini-1.5-flash`).
3. Run the **smoke test** cell before the big loop.
4. Use a key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (not a local proxy).
5. If `tornado.access` + `::1` appears, you are hitting **localhost** — disable VPN/proxy or point Colab at Google directly.
6. Enable billing if your region blocks the free tier.

Start with `LIMIT = 3` and one city file (`en_narrative_fill_smolensk.json`).